# C++ vs Python Performance Benchmark

This notebook demonstrates the performance advantage of using C++ for compute-intensive strategy logic.
We compare the execution time of a Moving Average Crossover strategy implemented in pure Python versus optimized C++ using pybind11.

In [ ]:
import time
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.abspath(".."))

import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
from src.events import MarketEvent
from src.strategy import CppMovingAverageCrossStrategy, MovingAverageCrossStrategy

# Set nicer plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Setup Benchmark Data
We generate 100,000 synthetic market events to simulate a high-frequency trading scenario.

In [ ]:
num_events = 100_000
prices = [100.0 + random.uniform(-1, 1) for _ in range(num_events)]
events = [MarketEvent(datetime.now(), "BENCH", p, 100) for p in prices]

print(f"Generated {num_events} market events.")

## 2. Python Implementation
Standard Python implementation using Python objects and lists.

In [ ]:
py_strategy = MovingAverageCrossStrategy(short_window=50, long_window=200)

start_time = time.time()
for e in events:
    py_strategy.calculate_signals(e)
py_duration = time.time() - start_time

print(f"Python Implementation: {py_duration:.4f} seconds")

## 3. C++ Implementation (Fast Path)
Optimized C++ implementation bound with `pybind11`. We use the "Fast Path" which passes raw floats to C++ to avoid overhead.

In [ ]:
if CppMovingAverageCrossStrategy is None:
    print("C++ Implementation not available.")
else:
    cpp_strategy_fast = CppMovingAverageCrossStrategy(short_window=50, long_window=200)
    
    start_time = time.time()
    for e in events:
        # Pass only price (simple float) to avoid Python object overhead
        cpp_strategy_fast.calculate_signal_fast(e.price)
    cpp_duration = time.time() - start_time
    
    print(f"C++ Fast Path: {cpp_duration:.4f} seconds")

## 4. Results & Visualization

In [ ]:
if CppMovingAverageCrossStrategy is not None:
    speedup = py_duration / cpp_duration
    print(f"Speedup: {speedup:.2f}x")
    
    # Visualization
    results = pd.DataFrame({
        'Implementation': ['Python', 'C++ (Fast Path)'],
        'Time (s)': [py_duration, cpp_duration]
    })
    
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(x='Implementation', y='Time (s)', data=results, palette='viridis')
    plt.title(f'Performance Benchmark ({num_events:,} events)', fontsize=16)
    plt.ylabel('Execution Time (seconds)', fontsize=12)
    
    # Add value labels
    for i, v in enumerate(results['Time (s)']):
        ax.text(i, v + (max(results['Time (s)'])*0.02), f'{v:.4f}s', ha='center', fontsize=12, fontweight='bold')
        
    plt.show()